# Course 4: Large Language Models
## Session 11: Evaluation, interpretability and safety

**Goal:** build the tooling for a 100-question Bangla evaluation set, then train a small sparse autoencoder on a tiny model's activations.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) with tiny models and a small synthetic Bangla corpus. Cells labelled NOT RUN need a GPU, a model download or an account; run them on Kaggle or Colab.

## Part A: an evaluation set you can trust
### 1. A fixed format and a validator
Write questions in JSON Lines, one per line. Validate before anything else: most evaluation bugs are data bugs.

In [ ]:
import json, random, unicodedata, numpy as np
from collections import Counter
EXAMPLES = [
 {"id": "geo-001", "category": "geography", "question": "বাংলাদেশের রাজধানী কোনটি?", "choices": ["চট্টগ্রাম", "ঢাকা", "খুলনা", "সিলেট"], "answer": 1, "source": "written by the course team"},
 {"id": "math-001", "category": "arithmetic", "question": "১২ + ৭ = ?", "choices": ["১৮", "১৯", "২০", "২১"], "answer": 1, "source": "written by the course team"},
 {"id": "lang-001", "category": "language", "question": "'সকাল' শব্দের বিপরীত শব্দ কোনটি?", "choices": ["দুপুর", "রাত", "সন্ধ্যা", "ভোর"], "answer": 2, "source": "written by the course team"},
 {"id": "sci-001", "category": "science", "question": "পানির রাসায়নিক সংকেত কোনটি?", "choices": ["CO2", "O2", "H2O", "NaCl"], "answer": 2, "source": "written by the course team"},
]
def validate(items, min_per_category=1):
    problems, ids = [], Counter(i["id"] for i in items)
    for it in items:
        if ids[it["id"]] > 1: problems.append(f"{it['id']}: duplicate id")
        if len(it["choices"]) != 4 or len(set(it["choices"])) != 4: problems.append(f"{it['id']}: needs 4 different choices")
        if not 0 <= it["answer"] < 4: problems.append(f"{it['id']}: answer index out of range")
        if it["question"] != unicodedata.normalize("NFC", it["question"]): problems.append(f"{it['id']}: not NFC-normalized")
        if not it.get("source"): problems.append(f"{it['id']}: no source")
    q = Counter(unicodedata.normalize("NFC", i["question"]).strip() for i in items)
    problems += [f"duplicate question: {k}" for k, v in q.items() if v > 1]
    pos = Counter(i["answer"] for i in items)
    if items and max(pos.values()) > 0.4 * len(items) and len(items) >= 20: problems.append(f"answer positions are unbalanced: {dict(pos)}")
    return problems
print("problems found:", validate(EXAMPLES))
bad = EXAMPLES + [dict(EXAMPLES[0], id="geo-001"), dict(EXAMPLES[1], id="math-002", answer=5)]
for p in validate(bad): print("  ", p)
assert validate(EXAMPLES) == [] and len(validate(bad)) >= 3

### 2. Scoring with an honest error bar
With 100 questions, a score of 62% has a 95% interval of roughly ±10 points. Two models 4 points apart are not distinguishable. Compare models on the same questions with a paired bootstrap.

In [ ]:
def accuracy_ci(correct, n_boot=2000, seed=0):
    correct = np.asarray(correct, dtype=float); rng = np.random.default_rng(seed)
    boots = rng.choice(correct, (n_boot, len(correct))).mean(1)
    return correct.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)

def paired_difference(a, b, n_boot=2000, seed=0):
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float); rng = np.random.default_rng(seed)
    boots = rng.choice(d, (n_boot, len(d))).mean(1)
    return d.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)

rng = np.random.default_rng(0)
model_a = rng.random(100) < 0.62; model_b = np.where(rng.random(100) < 0.85, model_a, ~model_a)     # two simulated models that mostly agree
print("model A: %.2f  (95%% interval %.2f to %.2f)" % accuracy_ci(model_a))
print("model B: %.2f  (95%% interval %.2f to %.2f)" % accuracy_ci(model_b))
print("A minus B: %+.2f  (95%% interval %+.2f to %+.2f)" % paired_difference(model_a, model_b))

### 3. Two checks every multiple-choice evaluation needs
- Position bias: shuffle the choices and score again. A model that prefers option A will change its score.
- Contamination: write the questions yourself, date them, and keep a private half. A question copied from the web may be in the training data.

In [ ]:
def shuffled(item, rng):
    order = list(range(4)); rng.shuffle(order)
    return dict(item, choices=[item["choices"][i] for i in order], answer=order.index(item["answer"]))
always_first = lambda item: 0                                   # a 'model' that always picks the first option
rng = random.Random(0)
items = [dict(EXAMPLES[i % 4], id=f"q{i}", answer=0, choices=[EXAMPLES[i % 4]["choices"][EXAMPLES[i % 4]["answer"]]] + [c for j, c in enumerate(EXAMPLES[i % 4]["choices"]) if j != EXAMPLES[i % 4]["answer"]]) for i in range(40)]
print("answers all in position 1: 'always first' scores %.2f" % np.mean([always_first(it) == it["answer"] for it in items]))
items2 = [shuffled(it, rng) for it in items]
print("after shuffling the choices: it scores %.2f" % np.mean([always_first(it) == it["answer"] for it in items2]))

### 4. Run a real model on your set (NOT RUN here), and LLM-as-judge
For multiple choice, compare the model's log-probabilities of the four option letters; it is cheaper and cleaner than parsing generated text. For open answers, an LLM judge is convenient but has known biases (position, length, self-preference): always validate the judge against your own grading on a sample.

In [ ]:
# NOT RUN: needs a GPU and a model download.
# def predict(item, model, tok):
#     prompt = item["question"] + "\n" + "\n".join(f"{l}. {c}" for l, c in zip("ABCD", item["choices"])) + "\nউত্তর:"
#     logits = model(**tok(prompt, return_tensors="pt").to(model.device)).logits[0, -1]
#     return int(torch.stack([logits[tok.encode(" " + l, add_special_tokens=False)[-1]] for l in "ABCD"]).argmax())
# TODO: write 100 questions (25 in each of 4 categories), validate, evaluate two models, and report accuracy with intervals,
# accuracy by category, and the paired difference. Then re-run with shuffled choices.

## Part B: a sparse autoencoder
### 5. Collect activations from a tiny model

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
import random, itertools
SUBJ = ["মা", "বাবা", "শিক্ষক", "ছেলেটি", "মেয়েটি", "কৃষক", "দাদু", "রহিম"]
TIME = ["সকালে", "বিকেলে", "রাতে", "প্রতিদিন", "আজ"]
PLACE = ["বাজারে", "স্কুলে", "মাঠে", "বাড়িতে", "গ্রামে"]
OBJ_VERB = [("ভাত", "খায়"), ("বই", "পড়ে"), ("গান", "গায়"), ("ফুটবল", "খেলে"), ("চিঠি", "লেখে"), ("মাছ", "কেনে"), ("গল্প", "বলে"), ("ছবি", "আঁকে")]
VERB_OF = dict(OBJ_VERB)
# A toy language: subject, time, place, object, verb, full stop. The verb must fit the object (ভাত খায়, বই পড়ে).
ALL_SENTENCES = [[s, t, p, o, v, "।"] for s, t, p, (o, v) in itertools.product(SUBJ, TIME, PLACE, OBJ_VERB)]

def is_valid(words):
    return (len(words) == 6 and words[0] in SUBJ and words[1] in TIME and words[2] in PLACE
            and VERB_OF.get(words[3]) == words[4] and words[5] == "।")

EXTRA = ["<pad>", "<mask>", "প্রশ্ন", "উত্তর", ":", "?", "কী", "করে"]
VOCAB = EXTRA + SUBJ + TIME + PLACE + [o for o, _ in OBJ_VERB] + [v for _, v in OBJ_VERB] + ["।"]
STOI = {w: i for i, w in enumerate(VOCAB)}
enc = lambda words: [STOI[w] for w in words]
dec = lambda ids: [VOCAB[i] for i in ids]

In [ ]:
random.seed(0); torch.manual_seed(0)
_shuffled = ALL_SENTENCES[:]; random.shuffle(_shuffled)
train_sents, val_sents = _shuffled[:1200], _shuffled[1200:]            # 400 sentences the model never sees
train_ids = torch.tensor([i for s in train_sents for i in enc(s)])
val_ids = torch.tensor([i for s in val_sents for i in enc(s)])

def get_batch(data, block_size, batch_size):
    ix = torch.randint(0, (len(data) - block_size - 1) // 6, (batch_size,)) * 6      # start on a sentence boundary
    x = torch.stack([data[i:i + block_size] for i in ix]); y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

def train_lm(cfg, steps=600, lr=3e-3, data=None, seed=0):
    torch.manual_seed(seed); data = train_ids if data is None else data
    model = GPT(cfg); opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for _ in range(steps):
        x, y = get_batch(data, cfg.block_size, 32)
        _, loss = model(x, y); opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def val_loss(model, block_size=24):
    n = (len(val_ids) - 1) // block_size * block_size
    x = val_ids[:n].view(-1, block_size); y = val_ids[1:n + 1].view(-1, block_size)
    return model(x, y)[1].item()

@torch.no_grad()
def sample_sentences(model, n=200, temperature=1.0):
    prompt = torch.tensor([enc(train_sents[i]) for i in torch.randint(len(train_sents), (n,)).tolist()])   # one real sentence as the prompt
    out = model.generate(prompt, 6, temperature=temperature, top_p=1.0)[:, 6:]                     # the model writes the next one
    return [dec(row.tolist()) for row in out]

In [ ]:
cfg = Config(vocab_size=len(VOCAB), block_size=24, n_layer=2, n_head=2, d_model=64)
base = train_lm(cfg)
print("base model trained | validation loss %.3f | valid sentences in 200 samples: %.2f" % (val_loss(base), sum(map(is_valid, sample_sentences(base))) / 200))

In [ ]:
@torch.no_grad()
def collect(model, sents):
    x = torch.tensor([enc(s) for s in sents]); h = model(x, return_hidden=True)             # (sentences, 6, d_model)
    return h.reshape(-1, h.size(-1)), [w for s in sents for w in s], [i for s in sents for i in range(6)]
acts, act_words, act_pos = collect(base, train_sents)
mu = acts.mean(0, keepdim=True); acts_c = acts - mu
print("activations:", tuple(acts.shape))

### 6. Train the sparse autoencoder
Encode each activation into many more features than it has dimensions, keep only a few of them active (an L1 penalty), and decode back. The hope is that each feature stands for one thing a person can name.

In [ ]:
class SAE(nn.Module):
    def __init__(self, d, n_feat):
        super().__init__(); self.enc = nn.Linear(d, n_feat); self.dec = nn.Linear(n_feat, d)
    def forward(self, x):
        f = F.relu(self.enc(x)); return self.dec(f), f

torch.manual_seed(0)
sae = SAE(64, 256); opt = torch.optim.Adam(sae.parameters(), lr=2e-3)
for step in range(1500):
    b = acts_c[torch.randint(len(acts_c), (256,))]
    rec, f = sae(b)
    loss = ((rec - b) ** 2).sum(1).mean() + 0.3 * (f * sae.dec.weight.norm(dim=0)).sum(1).mean()
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    rec, f = sae(acts_c)
    explained = 1 - ((rec - acts_c) ** 2).sum() / (acts_c ** 2).sum(); l0 = (f > 1e-6).float().sum(1).mean(); alive = ((f > 1e-6).sum(0) > 0).sum()
print(f"variance explained {explained:.3f} | active features per token {l0:.1f} of 256 | features that ever fire {alive}")
assert explained > 0.8 and l0 < 40

### 7. Read the features
For each feature, look at the tokens where it fires most. A clean feature fires for one word, or one position, or one kind of word.

In [ ]:
with torch.no_grad(): _, f = sae(acts_c)
strength = f.sum(0); shown = 0
def purity(j):
    top = f[:, j].topk(30).indices.tolist(); w = Counter(act_words[i] for i in top); p = Counter(act_pos[i] for i in top)
    return w.most_common(1)[0], p.most_common(1)[0]
pure = 0
for j in strength.argsort(descending=True).tolist()[:40]:
    (word, wc), (pos, pc) = purity(j)
    if wc >= 27: pure += 1
    if shown < 8: print(f"feature {j:3d}: top word '{word}' in {wc}/30 of its strongest tokens | most common position {pos} ({pc}/30)"); shown += 1
print(f"{pure} of the 40 strongest features fire almost only on a single word")

Be careful what you conclude. The activations are taken after each word is read, and they are used to predict the next word, so a feature that fires 'on বই' may really encode 'the verb পড়ে comes next'. Naming a feature is a hypothesis. Test it by changing the feature and watching the model's output (TODO 3).

### 8. The dual use of these tools
The same methods that find a 'deception' feature could find and suppress a 'criticism of the government' feature. ICML 2026's outstanding position paper argues the alignment community is building a censor's toolkit without meaning to. When you publish a steering or detection method, say who could misuse it and how.

### TODO
1. Write your 100-question set and run section 4.
2. Sweep the L1 coefficient (0.03, 0.1, 0.3, 1.0) and plot variance explained against active features per token.
3. Steering: pick a feature that fires before one verb. Add 5 times its decoder direction to the hidden state at the object position and measure how the model's next-word probabilities change.